In [9]:
from pathlib import Path
import pandas as pd

In [13]:
DATA_PATH = Path("../data/combined_dataset_labeled.xlsx")

df = pd.read_excel(
    DATA_PATH,
    sheet_name="Research Dataset"
)

df.shape

(1055, 84)

In [14]:
df["Tissue_Grade_Source"].value_counts(dropna=False)

Tissue_Grade_Source
Eye-bank form    1037
NaN                18
Name: count, dtype: int64

In [15]:
pd.crosstab(
    df["Dataset_Part"],
    df["Tissue_Grade_Source"],
    dropna=False
)

Tissue_Grade_Source,Eye-bank form,NaN
Dataset_Part,,
Graded set (eye-bank forms),1037,18


In [16]:
df["Tissue_Grade"].value_counts(dropna=False)

Tissue_Grade
A+                                                                             543
A                                                                              311
B                                                                              167
NaN                                                                             19
A (release form; tissue sheet A+)                                                3
A (tissue sheet) / A+ (release form)                                             2
A (as read - verify)                                                             1
A+ / A (both circled on tissue sheet - verify)                                   1
B (release form A changed to B by hand; tissue sheet A)                          1
A+ (release; "-> B" handwritten)                                                 1
A+ (release form; handwritten mark may change to B - verify)                     1
A+ / A (both circled)                                                     

In [17]:
pd.crosstab(
    df["Tissue_Grade_Source"],
    df["Tissue_Grade"],
    dropna=False
)

Tissue_Grade,A,A (as read - verify),A (release form; tissue sheet A+),A (release form; tissue sheet circles A+),A (tissue sheet) / A+ (release form),A+,A+ (as read),A+ (release form; handwritten mark may change to B - verify),"A+ (release; ""-> B"" handwritten)",A+ / A (both circled on tissue sheet - verify),A+ / A (both circled),B,"B (printed A+ on release form, handwritten downgrade to B; tissue sheet A+)",B (release form A changed to B by hand; tissue sheet A),B (release grade A changed to B by hand),NaN
Tissue_Grade_Source,,,,,,,,,,,,,,,,
Eye-bank form,311,1,3,1,2,543,1,1,1,1,1,167,1,1,1,1
NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18


In [18]:
df["Needs_Review"].value_counts(dropna=False)

Needs_Review
NaN    1055
Name: count, dtype: int64

In [19]:
df.loc[
    df["Needs_Review"].notna(),
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source",
     "Needs_Review", "Review_Notes"]
].head(20)

,Tissue_ID,Tissue_Grade,Tissue_Grade_Source,Needs_Review,Review_Notes


In [20]:
grade_counts = (
    df["Tissue_Grade"]
    .value_counts(dropna=False)
    .reset_index()
)

grade_counts.columns = ["Tissue_Grade", "Count"]

grade_counts

,Tissue_Grade,Count
0,A+,543
1,A,311
2,B,167
3,NaN,19
4,A (release form; tissue sheet A+),3
5,A (tissue sheet) / A+ (release form),2
6,A (as read - verify),1
7,A+ / A (both circled on tissue sheet - verify),1
8,B (release form A changed to B by hand; tissue...,1
9,"A+ (release; ""-> B"" handwritten)",1


In [21]:
standard_grades = ["A+", "A", "B"]

non_standard = df[
    ~df["Tissue_Grade"].isin(standard_grades)
]

non_standard[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].sort_values("Tissue_Grade")

,Tissue_ID,Tissue_Grade,Tissue_Grade_Source
74,NEB-2020-10-12729 R,A (as read - verify),Eye-bank form
226,NEB-2024-05-16056 R,A (release form; tissue sheet A+),Eye-bank form
227,NEB-2024-05-16057 L,A (release form; tissue sheet A+),Eye-bank form
364,NEB-2024-08-16297 R,A (release form; tissue sheet A+),Eye-bank form
468,NEB-2025-09-17520 R,A (release form; tissue sheet circles A+),Eye-bank form
244,NEB-2024-05-16074 R,A (tissue sheet) / A+ (release form),Eye-bank form
319,NEB-2024-08-16252 L,A (tissue sheet) / A+ (release form),Eye-bank form
414,NEB-2025-09-17466 R,A+ (as read),Eye-bank form
321,NEB-2024-08-16254 L,A+ (release form; handwritten mark may change ...,Eye-bank form
302,NEB-2024-06-16133 L,"A+ (release; ""-> B"" handwritten)",Eye-bank form


In [22]:
non_standard[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].to_string(index=False)

'                   Tissue_ID                                                                Tissue_Grade Tissue_Grade_Source\n12346 R (year/month unknown)                                                                         NaN                 NaN\n         NEB-2020-09-12695 L                                                                         NaN                 NaN\n         NEB-2020-09-12697 L                                                                         NaN                 NaN\n         NEB-2020-09-12698 L                                                                         NaN                 NaN\n         NEB-2020-09-12699 R                                                                         NaN                 NaN\n         NEB-2020-09-12700 L                                                                         NaN                 NaN\n         NEB-2020-09-12701 R                                                                         NaN             

In [23]:
review_grades = {
    "12729": "A",
    "16056": "A",
    "16057": "A",
    "16060": "A",
    "16061": "B",
    "16074": "A+",
    "16133": "B",
    "16252": "A+",
    "16254": "B",
    "16297": "A",
    "16603": "A",
    "17466": "A",
    "17482": "B",
    "17495": "B",
    "17520": "A",
}

# Keep the original grade untouched
df["Tissue_Grade_Clean"] = df["Tissue_Grade"]

# Extract the numeric tissue ID from Tissue_ID
tissue_number = (
    df["Tissue_ID"]
    .astype(str)
    .str.extract(r"(\d{5})")[0]
)

# Apply specialist-reviewed grades
mask = tissue_number.isin(review_grades)

df.loc[mask, "Tissue_Grade_Clean"] = tissue_number[mask].map(review_grades)

In [24]:
df.loc[
    mask,
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Clean"]
].sort_values("Tissue_ID")

,Tissue_ID,Tissue_Grade,Tissue_Grade_Clean
74,NEB-2020-10-12729 R,A (as read - verify),A
226,NEB-2024-05-16056 R,A (release form; tissue sheet A+),A
227,NEB-2024-05-16057 L,A (release form; tissue sheet A+),A
230,NEB-2024-05-16060 R,A+ / A (both circled on tissue sheet - verify),A
231,NEB-2024-05-16061 L,B (release form A changed to B by hand; tissue...,B
244,NEB-2024-05-16074 R,A (tissue sheet) / A+ (release form),A+
302,NEB-2024-06-16133 L,"A+ (release; ""-> B"" handwritten)",B
319,NEB-2024-08-16252 L,A (tissue sheet) / A+ (release form),A+
321,NEB-2024-08-16254 L,A+ (release form; handwritten mark may change ...,B
364,NEB-2024-08-16297 R,A (release form; tissue sheet A+),A


In [25]:
df["Tissue_Grade_Clean"].value_counts(dropna=False)

Tissue_Grade_Clean
A+     545
A      319
B      172
NaN     19
Name: count, dtype: int64

In [26]:
missing_grades = df[
    df["Tissue_Grade_Clean"].isna()
]

missing_grades[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].to_string(index=False)

'                   Tissue_ID Tissue_Grade Tissue_Grade_Source\n12346 R (year/month unknown)          NaN                 NaN\n         NEB-2020-09-12695 L          NaN                 NaN\n         NEB-2020-09-12697 L          NaN                 NaN\n         NEB-2020-09-12698 L          NaN                 NaN\n         NEB-2020-09-12699 R          NaN                 NaN\n         NEB-2020-09-12700 L          NaN                 NaN\n         NEB-2020-09-12701 R          NaN                 NaN\n         NEB-2020-10-12761 R          NaN                 NaN\n         NEB-2020-10-12763 L          NaN                 NaN\n         NEB-2020-10-12772 L          NaN                 NaN\n         NEB-2022-04-13848 R          NaN                 NaN\n14302 L (year/month unknown)          NaN                 NaN\n         NEB-2024-06-16116 L          NaN                 NaN\n         NEB-2024-06-16117 R          NaN                 NaN\n         NEB-2024-06-16137 L          NaN             

In [29]:
# Final grading summary
grade_summary = (
    df["Tissue_Grade_Clean"]
    .value_counts(dropna=False)
    .rename_axis("Tissue_Grade_Clean")
    .reset_index(name="Count")
)

print(grade_summary)

  Tissue_Grade_Clean  Count
0                 A+    545
1                  A    319
2                  B    172
3                NaN     19


In [33]:
print(df["ECD_CD_cells_per_mm2"].describe())

count     956.000000
mean     2581.553347
std       468.981048
min         0.000000
25%      2336.000000
50%      2591.000000
75%      2857.000000
max      4484.000000
Name: ECD_CD_cells_per_mm2, dtype: float64


In [34]:
print((df["ECD_CD_cells_per_mm2"] == 0).sum())

4


In [35]:
print(df.loc[
    df["ECD_CD_cells_per_mm2"] == 0,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count"
    ]
])

               Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2   CV  \
677  NEB-2023-03-14825 R                  B                   0.0  0.0   
818  NEB-2023-05-15117 R                  B                   0.0  0.0   
833  NEB-2023-06-15153 R                  B                   0.0  0.0   
890  NEB-2023-07-15267 R                  B                   0.0  0.0   

     HEX_percent   SD  Total_Cell_Count  
677          0.0  0.0               0.0  
818          0.0  0.0               0.0  
833          0.0  0.0               0.0  
890          0.0  0.0               0.0  


In [38]:
konan_measurement_cols = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2"
]

invalid_konan = df["ECD_CD_cells_per_mm2"] == 0

df.loc[invalid_konan, konan_measurement_cols] = pd.NA

print(invalid_konan.sum())

0


In [39]:
df.loc[
    invalid_konan,
    ["Tissue_ID", "Tissue_Grade_Clean"] + konan_measurement_cols
]

,Tissue_ID,Tissue_Grade_Clean,ECD_CD_cells_per_mm2,CV,HEX_percent,SD,Total_Cell_Count,Total_Area_um2


In [40]:
df.loc[
    df["Tissue_ID"].isin([
        "NEB-2023-03-14825 R",
        "NEB-2023-05-15117 R",
        "NEB-2023-06-15153 R",
        "NEB-2023-07-15267 R"
    ]),
    ["Tissue_ID", "Tissue_Grade_Clean"] + konan_measurement_cols
]

,Tissue_ID,Tissue_Grade_Clean,ECD_CD_cells_per_mm2,CV,HEX_percent,SD,Total_Cell_Count,Total_Area_um2
677,NEB-2023-03-14825 R,B,NaN,NaN,NaN,NaN,NaN,NaN
818,NEB-2023-05-15117 R,B,NaN,NaN,NaN,NaN,NaN,NaN
833,NEB-2023-06-15153 R,B,NaN,NaN,NaN,NaN,NaN,NaN
890,NEB-2023-07-15267 R,B,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
print((df[konan_measurement_cols] == 0).sum())


ECD_CD_cells_per_mm2    0
CV                      0
HEX_percent             0
SD                      0
Total_Cell_Count        0
Total_Area_um2          0
dtype: int64


In [45]:
leakage_candidates = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings"
]

for col in leakage_candidates:
    print(f"\n--- {col} ---")
    print(df[col].dropna().value_counts().head(10))


--- Overall_Donor_Quality ---
Overall_Donor_Quality
Excellent                                                          137
Very good                                                           79
Good                                                                31
Very good (as read)                                                  4
Excellent (as read)                                                  2
Good (as read)                                                       2
Good (Very good struck)                                              2
Excellent / very good (mark spans both - verify)                     2
Very good (Good also marked - verify)                                1
Very good (stroke runs across Excellent and Very good - verify)      1
Name: count, dtype: int64

--- Transplant_Suitability ---
Transplant_Suitability
Suitable (released for surgery - printed release form)                                                       197
Suitable                                   

In [46]:
leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes"
]

len(leakage_columns)

15

In [48]:
grade_mentions = df["Additional_Findings"].dropna()[
    df["Additional_Findings"].dropna().str.contains(
        r"\bGrade\s*(A\+|A|B)\b",
        case=False,
        regex=True,
        na=False
    )
]

print(grade_mentions)

156    Release form: Grade A+, irregular cut; rim 2.5...
157    Release form: Grade A+, irregular cut; rim 2.5...
165    Release form: Grade A+, irregular cut, pterygi...
173    Release form: Grade A, irregular cut, mild ict...
175    Release form: Grade A, irregular cut, mild ict...
                             ...                        
574    Release form: Grade A+, irregular cut; rim 3.0...
576    Release form: Grade B, irregular cut; rim 3.00...
584    Release form: Grade A, irregular cut; rim 2.00...
585    Release form: Grade A+, irregular cut; rim 3.0...
589    Release form: Grade A+, irregular cut, eye las...
Name: Additional_Findings, Length: 255, dtype: str


C:\Users\pulit\AppData\Local\Temp\ipykernel_2836\1925329226.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Additional_Findings"].dropna().str.contains(


In [49]:
leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings"
]

In [50]:
len(leakage_columns)

16

In [51]:
numeric_cols = df.select_dtypes(include="number").columns


In [54]:
print(df[numeric_cols].corr().round(2))

                          ECD_CD_cells_per_mm2    CV  HEX_percent    SD  \
ECD_CD_cells_per_mm2                      1.00 -0.04         0.02 -0.75   
CV                                       -0.04  1.00        -0.51  0.53   
HEX_percent                               0.02 -0.51         1.00 -0.28   
SD                                       -0.75  0.53        -0.28  1.00   
Total_Cell_Count                          0.26 -0.04         0.15 -0.33   
Total_Area_um2                           -0.64  0.03         0.09  0.42   
Donor_Age                                -0.20  0.12        -0.03  0.17   
Tissue_Serial                            -0.06  0.02        -0.02 -0.00   
PDF_Pages                                 0.04 -0.02        -0.04 -0.08   
Konan_Mean_Cell_Area_um2                 -0.89  0.10        -0.06  0.88   
Death_to_Surgery_Days                    -0.21  0.10        -0.09  0.24   
Request_to_Surgery_Days                   0.07 -0.17         0.11 -0.17   
Needs_Review             

In [55]:
# Inspect numerical columns

print(df[numeric_cols].describe())

       ECD_CD_cells_per_mm2          CV  HEX_percent          SD  \
count            952.000000  952.000000   952.000000  951.000000   
mean            2592.400210   38.130252    54.032563  153.001052   
std              438.998119    4.993776     6.437436   47.701319   
min              676.000000   24.000000    27.000000   86.000000   
25%             2336.000000   35.000000    50.000000  127.000000   
50%             2591.000000   38.000000    54.000000  144.000000   
75%             2857.000000   41.000000    58.000000  164.000000   
max             4484.000000   74.000000    76.000000  753.000000   

       Total_Cell_Count  Total_Area_um2    Donor_Age  Tissue_Serial  \
count        952.000000      952.000000  1050.000000    1055.000000   
mean         107.018908    42265.138655    59.672381   15353.921327   
std           16.351349     9122.697304    10.901732    1462.541930   
min           10.000000     5071.000000    15.000000   12346.000000   
25%          104.000000    37512

In [56]:
df["Needs_Review"].isna().all()

np.True_

In [58]:
unusable_columns = [
    "Needs_Review"
]

In [59]:
text_numeric_candidates = [
    "Form_ECD_cells_per_mm2",
    "Recipient_Age",
    "Donor_Graft_Size_mm",
    "Recipient_Bed_Size_mm",
    "Previous_Grafts"
]

for col in text_numeric_candidates:
    print(f"\n--- {col} ---")
    print(df[col].dropna().head(15).to_list())


--- Form_ECD_cells_per_mm2 ---
[2132, 2212, 2538, 2584, 3115, 2950, 2993, 3049, 2833, 2780, 3049, 2710, 2625, 2833, 3012]

--- Recipient_Age ---
[48, 64, 53, 57, '2 y 3 m', '35', 80, '60 (as read)', 68, 47, 14, 69, '72 (as read)', 55, 21]

--- Donor_Graft_Size_mm ---
[8.25, 8, 8.5, 7.5, 8.5, 9, 7, 11, 7.5, 8, 8, '8-8.5', '8 (as read)', 8.25, 8]

--- Recipient_Bed_Size_mm ---
[8, 8.5, 8, 8, 8.25, 7.5, 7, 8.5, 8.25, '8 (as read)', '8 (as read)', 7.5, 9, '11.5 (as read)', 8.5]

--- Previous_Grafts ---
['1', '2', '1', 'Not ticked (diagnosis implies >=1)', '1', '1', 'Not ticked', '1', 'Not ticked', '>=1 (repeat PK)', 0, 1, 0, 1, 2]


In [60]:
df["Form_ECD_cells_per_mm2"] = pd.to_numeric(
    df["Form_ECD_cells_per_mm2"],
    errors="coerce"
)

df["Form_ECD_cells_per_mm2"].dtype

dtype('float64')

In [62]:
print(df["Form_ECD_cells_per_mm2"].describe())

count     575.000000
mean     2659.478261
std       464.859121
min       676.000000
25%      2392.000000
50%      2639.000000
75%      2937.000000
max      4484.000000
Name: Form_ECD_cells_per_mm2, dtype: float64


In [64]:
age_text_values = df.loc[
    df["Recipient_Age"].notna() &
    ~df["Recipient_Age"].astype(str).str.fullmatch(r"\d+(\.\d+)?"),
    "Recipient_Age"
]

print(age_text_values.value_counts())

Recipient_Age
60 (as read)                                           2
17 (as read)                                           2
21 (as read)                                           2
2 y 3 m                                                1
72 (as read)                                           1
18 (as read)                                           1
13 (as read)                                           1
67 (as read)                                           1
6/12 (as read - possibly 6 months)                     1
30 (as read)                                           1
9.5 (written "9 1/2 yr")                               1
68 (report) / 72 (request)                             1
71 (as read)                                           1
3 y 5 m (written "3 5/12")                             1
64 (request; transplant report 60 as read)             1
31 (as read)                                           1
53 (request; transplant report 52)                     1
76 (request; tran

In [66]:
age_conflicts = df.loc[
    df["Recipient_Age"].notna() &
    df["Recipient_Age"].astype(str).str.contains(
        r"/|request.*report|report.*request",
        case=False,
        regex=True,
        na=False
    ),
    ["Tissue_ID", "Recipient_Age"]
]

print(age_conflicts)

               Tissue_ID                                      Recipient_Age
44   NEB-2020-09-12697 L                 6/12 (as read - possibly 6 months)
79   NEB-2020-10-12734 L                           9.5 (written "9 1/2 yr")
84   NEB-2020-10-12739 L                         68 (report) / 72 (request)
242  NEB-2024-05-16072 L                         3 y 5 m (written "3 5/12")
275  NEB-2024-05-16105 R         64 (request; transplant report 60 as read)
287  NEB-2024-06-16118 L                 53 (request; transplant report 52)
291  NEB-2024-06-16122 R                 76 (request; transplant report 74)
297  NEB-2024-06-16128 L         57 (request; transplant report 52 as read)
322  NEB-2024-08-16255 R                69 (report) / 65 (request, as read)
329  NEB-2024-08-16262 L                         78 (report) / 70 (request)
347  NEB-2024-08-16280 R               44 (request); 43 (transplant report)
355  NEB-2024-08-16288 R  46 (transplant report); request reads 40 (verify)
361  NEB-202

In [67]:
df["Recipient_Age_Clean"] = df["Recipient_Age"]

In [68]:
verified_recipient_ages = {
    "NEB-2020-09-12697 L": 21,
    "NEB-2020-10-12739 L": 58,
    "NEB-2024-05-16072 L": 35,
    "NEB-2024-05-16105 R": 52,
    "NEB-2024-06-16118 L": 58,
    "NEB-2024-06-16122 R": 38,
    "NEB-2024-06-16128 L": 19,
    "NEB-2024-08-16255 R": 64,
    "NEB-2024-08-16262 L": 56,
    "NEB-2024-08-16280 R": 41,
    "NEB-2024-08-16288 R": 60,
    "NEB-2024-08-16294 L": 55,
    "NEB-2025-09-17530 R": 62,
    "NEB-2025-10-17533 R": 45,
    "NEB-2025-11-17625 L": 60,
    "NEB-2025-11-17628 R": 62,
}

In [69]:
df.loc[
    df["Tissue_ID"].isin(verified_recipient_ages),
    "Recipient_Age_Clean"
] = df.loc[
    df["Tissue_ID"].isin(verified_recipient_ages),
    "Tissue_ID"
].map(verified_recipient_ages)

In [71]:
print(df.loc[
    df["Tissue_ID"].isin(verified_recipient_ages),
    ["Tissue_ID", "Recipient_Age", "Recipient_Age_Clean"]
])

               Tissue_ID                                      Recipient_Age  \
44   NEB-2020-09-12697 L                 6/12 (as read - possibly 6 months)   
84   NEB-2020-10-12739 L                         68 (report) / 72 (request)   
242  NEB-2024-05-16072 L                         3 y 5 m (written "3 5/12")   
275  NEB-2024-05-16105 R         64 (request; transplant report 60 as read)   
287  NEB-2024-06-16118 L                 53 (request; transplant report 52)   
291  NEB-2024-06-16122 R                 76 (request; transplant report 74)   
297  NEB-2024-06-16128 L         57 (request; transplant report 52 as read)   
322  NEB-2024-08-16255 R                69 (report) / 65 (request, as read)   
329  NEB-2024-08-16262 L                         78 (report) / 70 (request)   
347  NEB-2024-08-16280 R               44 (request); 43 (transplant report)   
355  NEB-2024-08-16288 R  46 (transplant report); request reads 40 (verify)   
361  NEB-2024-08-16294 L               64 (request);

In [73]:
remaining_age_values = df.loc[
    df["Recipient_Age_Clean"].notna() &
    ~df["Recipient_Age_Clean"].astype(str).str.fullmatch(r"\d+(\.\d+)?"),
    ["Tissue_ID", "Recipient_Age", "Recipient_Age_Clean"]
]

print(remaining_age_values)

               Tissue_ID             Recipient_Age       Recipient_Age_Clean
12   NEB-2020-09-12665 L                   2 y 3 m                   2 y 3 m
18   NEB-2020-09-12671 L              60 (as read)              60 (as read)
23   NEB-2020-09-12676 L              72 (as read)              72 (as read)
28   NEB-2020-09-12681 L              18 (as read)              18 (as read)
34   NEB-2020-09-12687 L              13 (as read)              13 (as read)
37   NEB-2020-09-12690 L              67 (as read)              67 (as read)
73   NEB-2020-10-12728 L              30 (as read)              30 (as read)
79   NEB-2020-10-12734 L  9.5 (written "9 1/2 yr")  9.5 (written "9 1/2 yr")
104  NEB-2020-11-12806 R              60 (as read)              60 (as read)
124  NEB-2020-11-12829 R              71 (as read)              71 (as read)
284  NEB-2024-06-16115 L              31 (as read)              31 (as read)
443  NEB-2025-09-17495 R              17 (as read)              17 (as read)

In [74]:
import re
import pandas as pd

# Create cleaned age column from the original source column
df["Recipient_Age_Clean"] = df["Recipient_Age"]

# Manually verified recipient ages
verified_recipient_ages = {
    "NEB-2020-09-12697 L": 21,
    "NEB-2020-10-12739 L": 58,
    "NEB-2024-05-16072 L": 35,
    "NEB-2024-05-16105 R": 52,
    "NEB-2024-06-16118 L": 58,
    "NEB-2024-06-16122 R": 38,
    "NEB-2024-06-16128 L": 19,
    "NEB-2024-08-16255 R": 64,
    "NEB-2024-08-16262 L": 56,
    "NEB-2024-08-16280 R": 41,
    "NEB-2024-08-16288 R": 60,
    "NEB-2024-08-16294 L": 55,
    "NEB-2025-09-17530 R": 62,
    "NEB-2025-10-17533 R": 45,
    "NEB-2025-11-17625 L": 60,
    "NEB-2025-11-17628 R": 62,
    "NEB-2025-10-17606 R": 56,
}

# Apply manually verified ages
for tissue_id, age in verified_recipient_ages.items():
    df.loc[
        df["Tissue_ID"] == tissue_id,
        "Recipient_Age_Clean"
    ] = age


def clean_recipient_age(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    # Dataset notation:
    # "2 y 3 m" -> 23
    # "3 y 5 m" -> 35
    match = re.fullmatch(r"(\d+)\s*y\s*(\d+)\s*m.*", value)
    if match:
        return float(match.group(1) + match.group(2))

    # Extract first numeric value from straightforward annotations
    match = re.search(r"\d+(?:\.\d+)?", value)

    if match:
        return float(match.group())

    return pd.NA


# Clean values that were not manually verified
for idx in df.index:
    tissue_id = df.loc[idx, "Tissue_ID"]

    if tissue_id not in verified_recipient_ages:
        df.loc[idx, "Recipient_Age_Clean"] = clean_recipient_age(
            df.loc[idx, "Recipient_Age"]
        )


# 12734 L was not verified, so do not use the reported 9.5
df.loc[
    df["Tissue_ID"] == "NEB-2020-10-12734 L",
    "Recipient_Age_Clean"
] = pd.NA

# 17606 R was manually verified as 56
df.loc[
    df["Tissue_ID"] == "NEB-2025-10-17606 R",
    "Recipient_Age_Clean"
] = 56

In [77]:
print(df.loc[
    df["Recipient_Age_Clean"].notna() &
    ~df["Recipient_Age_Clean"].astype(str).str.fullmatch(r"\d+(\.\d+)?"),
    ["Tissue_ID", "Recipient_Age", "Recipient_Age_Clean"]
])

Empty DataFrame
Columns: [Tissue_ID, Recipient_Age, Recipient_Age_Clean]
Index: []


In [76]:
df.loc[
    df["Tissue_ID"].isin([
        "NEB-2020-09-12665 L",
        "NEB-2020-10-12734 L",
        "NEB-2025-10-17606 R",
        "NEB-2025-11-17628 R"
    ]),
    ["Tissue_ID", "Recipient_Age", "Recipient_Age_Clean"]
]

,Tissue_ID,Recipient_Age,Recipient_Age_Clean
12,NEB-2020-09-12665 L,2 y 3 m,23.0
79,NEB-2020-10-12734 L,"9.5 (written ""9 1/2 yr"")",<NA>
551,NEB-2025-10-17606 R,8 (as read - verify),56
570,NEB-2025-11-17628 R,66 (transplant report; request 60 - verify),62


In [78]:
print(df["Recipient_Age_Clean"].describe())

count     295.0
unique     78.0
top        60.0
freq       13.0
Name: Recipient_Age_Clean, dtype: float64


In [79]:
df["Recipient_Age_Clean"] = pd.to_numeric(
    df["Recipient_Age_Clean"],
    errors="coerce"
)

In [80]:
df["Recipient_Age_Clean"].dtype

dtype('float64')

In [81]:
print(df["Recipient_Age_Clean"].describe())

count    295.000000
mean      49.047458
std       21.209141
min        4.000000
25%       29.000000
50%       53.000000
75%       67.000000
max       85.000000
Name: Recipient_Age_Clean, dtype: float64


In [82]:
clinical_cols = [
    "Epithelium_Findings",
    "Epithelium_Severity",
    "Stroma_Findings",
    "Stroma_Severity",
    "Descemet_Findings",
    "Descemet_Severity",
    "Endothelium_Findings",
    "Endothelium_Severity",
    "Disease_Label",
    "Disease_Severity",
    "Primary_COD"
]

for col in clinical_cols:
    print(f"\n--- {col} ---")
    print(df[col].dropna().value_counts().head(15))


--- Epithelium_Findings ---
Epithelium_Findings
Clear and intact                                                                                         37
Nothing ticked                                                                                           17
Clear and intact ("mild arcus" annotation)                                                               16
Clear and intact. Release form: mild arcus, no exposure, no haziness, no ED                              13
Clear and intact ("mild arcus" annotation). Release form: mild arcus, no exposure, no haziness, no ED     9
Release form: mild arcus, mild exposure, no haziness, mild ED                                             8
Clear and intact. Release form: no arcus, no exposure, no haziness, no ED                                 8
Clear and intact; exposure                                                                                7
Exposure                                                                               

In [83]:
clinical_cols = [
    "Epithelium_Findings",
    "Epithelium_Severity",
    "Stroma_Findings",
    "Stroma_Severity",
    "Descemet_Findings",
    "Descemet_Severity",
    "Endothelium_Findings",
    "Endothelium_Severity",
    "Disease_Label",
    "Disease_Severity",
    "Primary_COD"
]

clinical_missing = (
    df[clinical_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(clinical_missing)

Endothelium_Severity    968
Epithelium_Severity     741
Stroma_Severity         640
Disease_Severity        510
Descemet_Severity       488
Endothelium_Findings    480
Primary_COD             474
Descemet_Findings       469
Epithelium_Findings     468
Stroma_Findings         467
Disease_Label             1
dtype: int64


In [84]:
clinical_feature_cols = [
    "Epithelium_Findings",
    "Stroma_Findings",
    "Descemet_Findings",
    "Endothelium_Findings",
]

missing_pattern = (
    df[clinical_feature_cols]
    .isna()
    .astype(int)
    .value_counts()
    .reset_index(name="Count")
)

print(missing_pattern)

   Epithelium_Findings  Stroma_Findings  Descemet_Findings  \
0                    0                0                  0   
1                    1                1                  1   
2                    0                0                  0   
3                    0                0                  1   
4                    1                0                  0   
5                    0                0                  1   

   Endothelium_Findings  Count  
0                     0    573  
1                     1    467  
2                     1     12  
3                     1      1  
4                     0      1  
5                     0      1  


In [86]:
clinical_feature_cols = [
    "Epithelium_Findings",
    "Stroma_Findings",
    "Descemet_Findings",
    "Endothelium_Findings",
]

df["Clinical_Findings_Available"] = (
    df[clinical_feature_cols].notna().all(axis=1)
).astype(int)

print(df["Clinical_Findings_Available"].value_counts())

Clinical_Findings_Available
1    573
0    482
Name: count, dtype: int64


In [88]:
print(df["Endothelium_Findings"].value_counts(dropna=False).head(30))

Endothelium_Findings
NaN                                                                                               480
Stress lines; central; few                                                                         96
Stress lines; central; few (release form: few stress lines)                                        78
Relative number of stress lines: few (release form: few stress lines)                              61
Relative number of stress lines: few                                                               52
Nothing ticked on evaluation form                                                                  28
Stress lines; peripheral; few                                                                      23
Release form: few stress lines                                                                     21
Stress lines; peripheral; few (release form: few stress lines)                                     19
Stress lines; central; few; cell drop-out mild               

In [89]:
df["Endothelium_Findings"].dropna().unique()

<StringArray>
[                                                                                                                                                    'Stress lines; central; few',
                                                                                                                                                  'Stress lines; peripheral; few',
                                                                                                                                                              'Stress lines; few',
                                                                                                                                          'Stress lines; central/peripheral; few',
                                                                                                                 'Stress lines; central/peripheral; cell drop-out mild (as read)',
                                                                                           

In [90]:
for col in [
    "Descemet_Findings",
    "Stroma_Findings",
    "Epithelium_Findings"
]:
    print(f"\n{col}: {df[col].dropna().nunique()} unique non-null values")


Descemet_Findings: 77 unique non-null values

Stroma_Findings: 306 unique non-null values

Epithelium_Findings: 410 unique non-null values


In [91]:
df["Stroma_Findings"].dropna().unique()

<StringArray>
[                                                                                                      'Clear; arcus',
                                                                                                              'Arcus',
                                                                                 'Arcus; diameter of clear area 9 mm',
                                                           'Clear; arcus ("mild arcus"); diameter of clear area 8 mm',
                                                         'Clear; arcus ("mild arcus"); diameter of clear area 8.5 mm',
                                                                        'Clear; arcus; diameter of clear area 8.5 mm',
                                                     'Clear; arcus ("moderate arcus"); diameter of clear area 7.5 mm',
                                                                        'Clear; arcus; diameter of clear area 7.5 mm',
                                  

In [92]:
df["Epithelium_Findings"].dropna().unique()

<StringArray>
[                                                                                                                'Clear and intact',
                                                                                           'Exposure; peripheral ("mild exposure")',
                                                                                       'Other: small superficial epithelial defect',
                                                                                                   'Other: small epithelial defect',
                                                                                                          'Other: mild arcus noted',
                                                                   'Haze; exposure; location peripheral; mild ED ("mild exposure")',
                                                                              'Clear and intact; exposure; other: ED 1/3 (as read)',
                                                       

In [94]:
print(pd.crosstab(
    df["Tissue_Grade_Clean"],
    df["Disease_Label"],
    margins=True
))

Disease_Label       FECD  FECD; Polymorphism and pleomorphism  \
Tissue_Grade_Clean                                              
A                      0                                    0   
A+                     1                                    0   
B                      0                                    2   
All                    1                                    2   

Disease_Label       FECD; Polymorphism and pleomorphism; Low cell density  \
Tissue_Grade_Clean                                                          
A                                                                   0       
A+                                                                  0       
B                                                                  88       
All                                                                88       

Disease_Label       FECD; Polymorphism and pleomorphism; Low cell density; Folds  \
Tissue_Grade_Clean                                            

In [98]:
print(df["Disease_Label"].value_counts(dropna=False))

Disease_Label
Normal                                                                   505
Folds                                                                    324
FECD; Polymorphism and pleomorphism; Low cell density                     88
Guttata                                                                   53
Guttata; Folds                                                            28
FECD; Polymorphism and pleomorphism; Low cell density; Folds              20
FECD; Polymorphism and pleomorphism; Low cell density; Guttata             8
Polymorphism and pleomorphism                                              5
Poor cornea                                                                5
Folds; Guttata                                                             3
FECD; Polymorphism and pleomorphism; Low cell density; Guttata; Folds      3
Guttata; FECD; Polymorphism and pleomorphism; Low cell density             2
FECD; Polymorphism and pleomorphism                           

In [99]:
import re
import pandas as pd

def extract_clear_area_diameter(value):
    if pd.isna(value):
        return pd.NA

    match = re.search(
        r"diameter of clear area(?:\s+as read)?\s*(\d+(?:\.\d+)?)\s*mm",
        str(value),
        flags=re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return pd.NA


df["Stroma_Clear_Area_Diameter_mm"] = (
    df["Stroma_Findings"]
    .apply(extract_clear_area_diameter)
)

In [101]:
print(df["Stroma_Clear_Area_Diameter_mm"].describe())

count     555.0
unique     17.0
top         8.0
freq      239.0
Name: Stroma_Clear_Area_Diameter_mm, dtype: float64


In [103]:
print(df[
    ["Stroma_Findings", "Stroma_Clear_Area_Diameter_mm"]
].dropna(
    subset=["Stroma_Clear_Area_Diameter_mm"]
).head(20))

                                      Stroma_Findings  \
5                  Arcus; diameter of clear area 9 mm   
6                  Arcus; diameter of clear area 9 mm   
7   Clear; arcus ("mild arcus"); diameter of clear...   
8   Clear; arcus ("mild arcus"); diameter of clear...   
9   Clear; arcus ("mild arcus"); diameter of clear...   
10        Clear; arcus; diameter of clear area 8.5 mm   
11  Clear; arcus ("mild arcus"); diameter of clear...   
12  Clear; arcus ("mild arcus"); diameter of clear...   
13  Clear; arcus ("moderate arcus"); diameter of c...   
14        Clear; arcus; diameter of clear area 7.5 mm   
15                 Arcus; diameter of clear area 8 mm   
16        Clear; arcus; diameter of clear area 7.5 mm   
17        Clear; arcus; diameter of clear area 7.5 mm   
18               Arcus; diameter of clear area 7.5 mm   
20  Arcus ("mild arcus"); diameter of clear area 9 mm   
21  Arcus ("mild arcus"); diameter of clear area 8 mm   
22  Arcus ("mild arcus"); diame

In [104]:
df["Stroma_Clear_Area_Diameter_mm"] = pd.to_numeric(
    df["Stroma_Clear_Area_Diameter_mm"],
    errors="coerce"
)

In [106]:
print(df["Stroma_Clear_Area_Diameter_mm"].describe())

count    555.000000
mean       8.417748
std        0.571562
min        6.000000
25%        8.000000
50%        8.500000
75%        9.000000
max       10.000000
Name: Stroma_Clear_Area_Diameter_mm, dtype: float64


In [107]:
def has_text(value, pattern):
    if pd.isna(value):
        return pd.NA
    return bool(re.search(pattern, str(value), flags=re.IGNORECASE))


df["Stroma_Is_Clear"] = df["Stroma_Findings"].apply(
    lambda x: has_text(x, r"\bclear\b")
)

df["Stroma_Is_Compact"] = df["Stroma_Findings"].apply(
    lambda x: has_text(x, r"\bcompact\b")
)

df["Stroma_Has_Arcus"] = df["Stroma_Findings"].apply(
    lambda x: has_text(x, r"\barcus\b")
)

df["Stroma_Has_Edema"] = df["Stroma_Findings"].apply(
    lambda x: has_text(x, r"\bedema\b")
)

df["Stroma_Has_Scar"] = df["Stroma_Findings"].apply(
    lambda x: has_text(x, r"\bscar\b")
)

In [109]:
print(df[
    [
        "Stroma_Is_Clear",
        "Stroma_Is_Compact",
        "Stroma_Has_Arcus",
        "Stroma_Has_Edema",
        "Stroma_Has_Scar"
    ]
].value_counts(dropna=False))

Stroma_Is_Clear  Stroma_Is_Compact  Stroma_Has_Arcus  Stroma_Has_Edema  Stroma_Has_Scar
NaN              NaN                NaN               NaN               NaN                467
True             False              True              False             False              214
                 True               True              False             False              146
                 False              True              True              False               74
                 True               False             False             False               46
                 False              False             False             False               29
                 True               True              True              False               25
                 False              True              False             True                13
                                    False             True              False                7
False            False              False             Tru

In [110]:
def detect_edema(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if re.search(r"\bno\s+edema\b", text):
        return False

    if re.search(r"\bedema\b", text):
        return True

    return False


df["Stroma_Has_Edema"] = df["Stroma_Findings"].apply(detect_edema)

In [112]:
print(df["Stroma_Has_Edema"].value_counts(dropna=False))

Stroma_Has_Edema
False    578
<NA>     467
True      10
Name: count, dtype: int64


In [113]:
def extract_arcus_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "arcus" not in text:
        return "None"

    if "moderate arcus" in text:
        return "Moderate"

    if "mild arcus" in text:
        return "Mild"

    if "level not circled" in text:
        return "Unspecified"

    return "Unspecified"


df["Stroma_Arcus_Severity"] = df["Stroma_Findings"].apply(
    extract_arcus_severity
)

In [115]:
print(df["Stroma_Arcus_Severity"].value_counts(dropna=False))

Stroma_Arcus_Severity
NaN            467
Unspecified    437
None           102
Mild            43
Moderate         6
Name: count, dtype: int64


In [116]:
df["Stroma_Scar_Type"] = pd.NA

mask = df["Stroma_Findings"].notna()

df.loc[
    mask & df["Stroma_Findings"].str.contains(
        "phaco scar", case=False, na=False
    ),
    "Stroma_Scar_Type"
] = "Phaco scar"

df.loc[
    mask
    & df["Stroma_Scar_Type"].isna()
    & df["Stroma_Findings"].str.contains(
        r"\bscar\b", case=False, na=False
    ),
    "Stroma_Scar_Type"
] = "Other scar"

df.loc[
    mask & df["Stroma_Scar_Type"].isna(),
    "Stroma_Scar_Type"
] = "None"

In [117]:
print(df["Stroma_Scar_Type"].value_counts(dropna=False))

Stroma_Scar_Type
None          555
<NA>          467
Other scar     27
Phaco scar      6
Name: count, dtype: int64


In [118]:
df["Descemet_Findings"].dropna().unique()

<StringArray>
[                                                                                'Folds',
                                                                   'Folds ("very mild")',
                                                       'Folds; relative number: several',
                                                 'Folds (mild and moderate both marked)',
                                                          'Folds; relative number: mild',
                                                    'Folds ("no folds" box also marked)',
                                    'Folds; relative number: several (mild overwritten)',
                                                  'Folds (mild struck, moderate ticked)',
                                              'Folds (box struck; mild marked - verify)',
                           'Folds (box struck through - as read); relative number: mild',
                                                                   'No folds or defect

In [120]:
def detect_descemet_review(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    review_terms = [
        "verify",
        "ambiguous",
        "both marked",
        "both ticked",
        "struck",
        "not ticked",
        "box not ticked",
        "box unticked",
    ]

    if any(term in text for term in review_terms):
        return True

    if "no folds" in text and "folds" in text:
        return True

    return False


df["Descemet_Review_Flag"] = df["Descemet_Findings"].apply(
    detect_descemet_review
)

In [122]:
print(df["Descemet_Review_Flag"].value_counts(dropna=False))

Descemet_Review_Flag
False    544
<NA>     469
True      42
Name: count, dtype: int64


In [123]:
def detect_folds(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    # Explicit negative finding
    if "no folds" in text and "folds" not in text.replace("no folds", ""):
        return False

    # Explicit positive finding
    if "folds" in text:
        return True

    # Severity marked but folds box not ticked
    if "folds box not ticked" in text or "folds box not ticked" in text:
        return False

    return False


df["Descemet_Has_Folds"] = df["Descemet_Findings"].apply(detect_folds)

In [124]:
df.loc[
    df["Descemet_Review_Flag"] == True,
    "Descemet_Has_Folds"
] = pd.NA

In [125]:
print(df["Descemet_Has_Folds"].value_counts(dropna=False))

Descemet_Has_Folds
True     535
<NA>     511
False      9
Name: count, dtype: int64


In [126]:
def extract_fold_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()
    if "verify" in text or "ambiguous" in text:
        return pd.NA

    if "moderate" in text:
        return "Moderate"

    if "very mild" in text:
        return "Very mild"

    if "mild" in text:
        return "Mild"

    return "Unspecified"


df["Descemet_Fold_Severity"] = df["Descemet_Findings"].apply(
    extract_fold_severity
)

df.loc[
    df["Descemet_Review_Flag"] == True,
    "Descemet_Fold_Severity"
] = pd.NA

In [128]:
print(df["Descemet_Fold_Severity"].value_counts(dropna=False))

Descemet_Fold_Severity
NaN            511
Mild           387
Unspecified    118
Moderate        21
Very mild       18
Name: count, dtype: int64


In [129]:
def extract_fold_relative_number(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "relative number" not in text:
        return "Unspecified"

    if "several/moderate" in text:
        return "Several/Moderate"

    if "several" in text:
        return "Several"

    if "moderate" in text:
        return "Moderate"

    if "mild" in text:
        return "Mild"

    return "Unspecified"


df["Descemet_Fold_Relative_Number"] = (
    df["Descemet_Findings"]
    .apply(extract_fold_relative_number)
)

df.loc[
    df["Descemet_Review_Flag"] == True,
    "Descemet_Fold_Relative_Number"
] = pd.NA

In [131]:
print(df["Descemet_Fold_Relative_Number"].value_counts(dropna=False))

Descemet_Fold_Relative_Number
NaN                 511
Unspecified         261
Mild                252
Several              28
Moderate              2
Several/Moderate      1
Name: count, dtype: int64


In [133]:
df["Endo_Has_Stress_Lines"] = df["Endothelium_Findings"].apply(
    lambda x: (
        pd.NA if pd.isna(x)
        else (
            False
            if re.search(r"no stress lines", str(x), flags=re.IGNORECASE)
            else bool(re.search(r"stress lines", str(x), flags=re.IGNORECASE))
        )
    )
)

print(df["Endo_Has_Stress_Lines"].value_counts(dropna=False))

Endo_Has_Stress_Lines
True     499
<NA>     480
False     76
Name: count, dtype: int64


In [134]:
def extract_stress_line_location(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "stress lines" not in text:
        return "None"

    if "central/peripheral" in text or "central and peripheral" in text:
        return "Central + Peripheral"

    if "central" in text:
        return "Central"

    if "peripheral" in text:
        return "Peripheral"

    return "Unspecified"


df["Endo_Stress_Line_Location"] = (
    df["Endothelium_Findings"]
    .apply(extract_stress_line_location)
)

In [136]:
print(df["Endo_Stress_Line_Location"].value_counts(dropna=False))

Endo_Stress_Line_Location
NaN                     480
Central                 243
Unspecified             174
Peripheral               67
None                     63
Central + Peripheral     28
Name: count, dtype: int64


In [137]:
def extract_stress_line_amount(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "no stress lines" in text:
        return "None"

    if "stress lines" not in text:
        return "None"

    if "several" in text:
        return "Several"

    if "few" in text:
        return "Few"

    return "Unspecified"


df["Endo_Stress_Line_Amount"] = (
    df["Endothelium_Findings"]
    .apply(extract_stress_line_amount)
)

In [138]:
print(df["Endo_Stress_Line_Amount"].value_counts(dropna=False))

Endo_Stress_Line_Amount
NaN            480
Few            479
None            76
Unspecified     18
Several          2
Name: count, dtype: int64


In [139]:
def detect_cell_dropout(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    return bool(
        re.search(r"cell\s*drop[- ]?out", text)
    )


df["Endo_Has_Cell_Dropout"] = (
    df["Endothelium_Findings"]
    .apply(detect_cell_dropout)
)

In [140]:
print(df["Endo_Has_Cell_Dropout"].value_counts(dropna=False))

Endo_Has_Cell_Dropout
False    482
<NA>     480
True      93
Name: count, dtype: int64


In [141]:
def detect_rbc(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    return bool(re.search(r"\brbc", text))


df["Endo_Has_RBC"] = (
    df["Endothelium_Findings"]
    .apply(detect_rbc)
)

In [143]:
print(df["Endo_Has_RBC"].value_counts(dropna=False))

Endo_Has_RBC
False    545
<NA>     480
True      30
Name: count, dtype: int64


In [144]:
def extract_rbc_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "rbc++" in text:
        return "++"

    if "rbc+" in text:
        return "+"

    if re.search(r"rbc[^a-z]*(mild)", text):
        return "Mild"

    if "rbc" in text:
        return "Unspecified"

    return "None"


df["Endo_RBC_Severity"] = (
    df["Endothelium_Findings"]
    .apply(extract_rbc_severity)
)

In [145]:
print(df["Endo_RBC_Severity"].value_counts(dropna=False))

Endo_RBC_Severity
None           545
NaN            480
Unspecified     22
+                6
++               2
Name: count, dtype: int64


In [146]:
def extract_cell_dropout_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "cell drop" not in text:
        return "None"

    if "moderate" in text:
        return "Moderate"

    if "mild" in text:
        return "Mild"

    return "Unspecified"


df["Endo_Cell_Dropout_Severity"] = (
    df["Endothelium_Findings"]
    .apply(extract_cell_dropout_severity)
)

In [147]:
print(df["Endo_Cell_Dropout_Severity"].value_counts(dropna=False))

Endo_Cell_Dropout_Severity
None        482
NaN         480
Mild         90
Moderate      3
Name: count, dtype: int64


In [148]:
endo_features = [
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
]

df[endo_features].describe(include="all").T

,count,unique,top,freq
Endo_Has_Stress_Lines,575,2,True,499
Endo_Stress_Line_Location,575,5,Central,243
Endo_Stress_Line_Amount,575,4,Few,479
Endo_Has_Cell_Dropout,575,2,False,482
Endo_Cell_Dropout_Severity,575,3,None,482
Endo_Has_RBC,575,2,False,545
Endo_RBC_Severity,575,4,None,545


In [149]:
for col in endo_features:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Endo_Has_Stress_Lines ---
Endo_Has_Stress_Lines
True     499
<NA>     480
False     76
Name: count, dtype: int64

--- Endo_Stress_Line_Location ---
Endo_Stress_Line_Location
NaN                     480
Central                 243
Unspecified             174
Peripheral               67
None                     63
Central + Peripheral     28
Name: count, dtype: int64

--- Endo_Stress_Line_Amount ---
Endo_Stress_Line_Amount
NaN            480
Few            479
None            76
Unspecified     18
Several          2
Name: count, dtype: int64

--- Endo_Has_Cell_Dropout ---
Endo_Has_Cell_Dropout
False    482
<NA>     480
True      93
Name: count, dtype: int64

--- Endo_Cell_Dropout_Severity ---
Endo_Cell_Dropout_Severity
None        482
NaN         480
Mild         90
Moderate      3
Name: count, dtype: int64

--- Endo_Has_RBC ---
Endo_Has_RBC
False    545
<NA>     480
True      30
Name: count, dtype: int64

--- Endo_RBC_Severity ---
Endo_RBC_Severity
None           545
NaN           

In [150]:
def detect_exposure(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    # Explicit negative
    if re.search(r"\bno exposure\b", text):
        return False

    if "exposure" in text:
        return True

    return False


df["Epi_Has_Exposure"] = (
    df["Epithelium_Findings"]
    .apply(detect_exposure)
)

In [152]:
print(df["Epi_Has_Exposure"].value_counts(dropna=False))

Epi_Has_Exposure
<NA>     468
True     316
False    271
Name: count, dtype: int64


In [153]:
def extract_exposure_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "no exposure" in text:
        return "None"

    if "very mild" in text and "exposure" in text:
        return "Very mild"

    if "mild exposure" in text:
        return "Mild"

    if "exposure" in text:
        return "Unspecified"

    return "None"


df["Epi_Exposure_Severity"] = (
    df["Epithelium_Findings"]
    .apply(extract_exposure_severity)
)

In [155]:
print(df["Epi_Exposure_Severity"].value_counts(dropna=False))

Epi_Exposure_Severity
NaN            468
None           271
Mild           157
Unspecified    142
Very mild       17
Name: count, dtype: int64


In [157]:
def detect_epithelial_defect(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if re.search(r"\bno\s+(epithelial\s+)?defect\b", text):
        return False

    if re.search(r"\b(epithelial\s+)?defect\b", text):
        return True

    return False


df["Epi_Has_Defect"] = df["Epithelium_Findings"].apply(
    detect_epithelial_defect
)

print(df["Epi_Has_Defect"].value_counts(dropna=False))

Epi_Has_Defect
False    583
<NA>     468
True       4
Name: count, dtype: int64


In [159]:
print(df.loc[
    df["Epi_Has_Defect"] == True,
    ["Tissue_ID", "Epithelium_Findings"]
])

              Tissue_ID                                Epithelium_Findings
5   NEB-2020-09-12658 R         Other: small superficial epithelial defect
6   NEB-2020-09-12659 L                     Other: small epithelial defect
18  NEB-2020-09-12671 L  Clear and intact; other: epithelial defect not...
19  NEB-2020-09-12672 R                      Other: ED (epithelial defect)


In [161]:
def detect_haze(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if re.search(r"\bno\s+haze\b", text):
        return False

    if re.search(r"\bhaze\b", text):
        return True

    return False


df["Epi_Has_Haze"] = df["Epithelium_Findings"].apply(
    detect_haze
)

print(df["Epi_Has_Haze"].value_counts(dropna=False))


Epi_Has_Haze
False    555
<NA>     468
True      32
Name: count, dtype: int64


In [163]:
print(df.loc[
    df["Epi_Has_Haze"] == True,
    ["Tissue_ID", "Epithelium_Findings"]
])

               Tissue_ID                                Epithelium_Findings
11   NEB-2020-09-12664 R  Haze; exposure; location peripheral; mild ED (...
29   NEB-2020-09-12682 R  Haze; exposure; other: superficial ED on the e...
51   NEB-2020-09-12704 L  Haze; exposure; location peripheral ("mild exp...
52   NEB-2020-09-12705 R  Haze; exposure; location peripheral; other: ED...
53   NEB-2020-10-12706 R                                  Haze; other: RBC+
54   NEB-2020-10-12707 L                             Haze; other: ED+, RBC+
79   NEB-2020-10-12734 L  Clear and intact; haze; exposure; location per...
97   NEB-2020-10-12761 R  Haze (mild); exposure (moderate); location cen...
99   NEB-2020-10-12772 L  Haze (very mild); exposure; peripheral ("ptery...
106  NEB-2020-11-12808 L  Clear and intact (stroke also crosses Haze box...
121  NEB-2020-11-12826 R                                               Haze
122  NEB-2020-11-12827 L                                               Haze
148  NEB-202

In [165]:
def extract_haze_severity(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "no haze" in text:
        return "None"

    if "very mild" in text and "haze" in text:
        return "Very mild"

    if re.search(r"haze\s*\(\s*mild\s*\)", text):
        return "Mild"

    if re.search(r"\bhaze\b[^;,.]*(?:[: ]|is )mild", text):
        return "Mild"

    if "haze" in text:
        return "Unspecified"

    return "None"


df["Epi_Haze_Severity"] = (
    df["Epithelium_Findings"]
    .apply(extract_haze_severity)
)

print(df["Epi_Haze_Severity"].value_counts(dropna=False))

Epi_Haze_Severity
None           555
NaN            468
Unspecified     28
Mild             3
Very mild        1
Name: count, dtype: int64


In [167]:
def extract_exposure_location(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    if "exposure" not in text:
        return "None"

    if "central/peripheral" in text or "central and peripheral" in text:
        return "Central + Peripheral"

    if "central" in text:
        return "Central"

    if "peripheral" in text:
        return "Peripheral"

    return "Unspecified"


df["Epi_Exposure_Location"] = (
    df["Epithelium_Findings"]
    .apply(extract_exposure_location)
)

print(df["Epi_Exposure_Location"].value_counts(dropna=False))

Epi_Exposure_Location
NaN                     468
Unspecified             266
None                    164
Peripheral               97
Central                  31
Central + Peripheral     29
Name: count, dtype: int64


In [168]:
def extract_exposure_location(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).lower()

    # Explicitly no exposure
    if re.search(r"\bno\s+exposure\b", text):
        return "None"

    # No exposure mentioned
    if "exposure" not in text:
        return "None"

    if "central/peripheral" in text or "central and peripheral" in text:
        return "Central + Peripheral"

    if "central" in text and "peripheral" in text:
        return "Central + Peripheral"

    if "central" in text:
        return "Central"

    if "peripheral" in text:
        return "Peripheral"

    return "Unspecified"


df["Epi_Exposure_Location"] = (
    df["Epithelium_Findings"]
    .apply(extract_exposure_location)
)

df["Epi_Exposure_Location"].value_counts(dropna=False)

Epi_Exposure_Location
NaN                     468
None                    271
Unspecified             159
Peripheral               97
Central + Peripheral     53
Central                   7
Name: count, dtype: int64

In [170]:
print(pd.crosstab(
    df["Epi_Has_Exposure"],
    df["Epi_Exposure_Location"],
    dropna=False
))

Epi_Exposure_Location  Central  Central + Peripheral  None  Peripheral  \
Epi_Has_Exposure                                                         
NaN                          0                     0     0           0   
False                        0                     0   271           0   
True                         7                    53     0          97   

Epi_Exposure_Location  Unspecified  NaN  
Epi_Has_Exposure                         
NaN                              0  468  
False                            0    0  
True                           159    0  


In [172]:
df["Epi_Has_Arcus"] = df["Epithelium_Findings"].apply(
    lambda x: (
        pd.NA
        if pd.isna(x)
        else bool(re.search(r"\barcus\b", str(x), flags=re.IGNORECASE))
    )
)

print(df["Epi_Has_Arcus"].value_counts(dropna=False))

Epi_Has_Arcus
<NA>     468
True     328
False    259
Name: count, dtype: int64


In [174]:
print(df.loc[
    df["Epithelium_Findings"].notna(),
    "Epithelium_Findings"
].value_counts().head(30))

Epithelium_Findings
Clear and intact                                                                                         37
Nothing ticked                                                                                           17
Clear and intact ("mild arcus" annotation)                                                               16
Clear and intact. Release form: mild arcus, no exposure, no haziness, no ED                              13
Clear and intact ("mild arcus" annotation). Release form: mild arcus, no exposure, no haziness, no ED     9
Release form: mild arcus, mild exposure, no haziness, mild ED                                             8
Clear and intact. Release form: no arcus, no exposure, no haziness, no ED                                 8
Clear and intact; exposure                                                                                7
Exposure                                                                                                  7
Exposure

In [175]:
epi_features = [
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
]

df[epi_features].info()

<class 'pandas.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Epi_Has_Exposure       587 non-null    object
 1   Epi_Exposure_Severity  587 non-null    str   
 2   Epi_Exposure_Location  587 non-null    str   
 3   Epi_Has_Defect         587 non-null    object
 4   Epi_Has_Haze           587 non-null    object
 5   Epi_Haze_Severity      587 non-null    str   
 6   Epi_Has_Arcus          587 non-null    object
dtypes: object(4), str(3)
memory usage: 57.8+ KB


In [177]:
print(df[epi_features].isna().sum())

Epi_Has_Exposure         468
Epi_Exposure_Severity    468
Epi_Exposure_Location    468
Epi_Has_Defect           468
Epi_Has_Haze             468
Epi_Haze_Severity        468
Epi_Has_Arcus            468
dtype: int64


In [179]:
stroma_features = [
    "Stroma_Clear_Area_Diameter_mm",
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
]

print(df[stroma_features].isna().sum())

Stroma_Clear_Area_Diameter_mm    500
Stroma_Is_Clear                  467
Stroma_Is_Compact                467
Stroma_Has_Arcus                 467
Stroma_Has_Edema                 467
Stroma_Arcus_Severity            467
Stroma_Scar_Type                 467
dtype: int64


In [182]:
print(df.loc[
    df["Stroma_Clear_Area_Diameter_mm"].notna(),
    ["Tissue_ID", "Stroma_Findings", "Stroma_Clear_Area_Diameter_mm"]
].head(20))

              Tissue_ID                                    Stroma_Findings  \
5   NEB-2020-09-12658 R                 Arcus; diameter of clear area 9 mm   
6   NEB-2020-09-12659 L                 Arcus; diameter of clear area 9 mm   
7   NEB-2020-09-12660 L  Clear; arcus ("mild arcus"); diameter of clear...   
8   NEB-2020-09-12661 R  Clear; arcus ("mild arcus"); diameter of clear...   
9   NEB-2020-09-12662 L  Clear; arcus ("mild arcus"); diameter of clear...   
10  NEB-2020-09-12663 R        Clear; arcus; diameter of clear area 8.5 mm   
11  NEB-2020-09-12664 R  Clear; arcus ("mild arcus"); diameter of clear...   
12  NEB-2020-09-12665 L  Clear; arcus ("mild arcus"); diameter of clear...   
13  NEB-2020-09-12666 L  Clear; arcus ("moderate arcus"); diameter of c...   
14  NEB-2020-09-12667 R        Clear; arcus; diameter of clear area 7.5 mm   
15  NEB-2020-09-12668 L                 Arcus; diameter of clear area 8 mm   
16  NEB-2020-09-12669 R        Clear; arcus; diameter of clear a

In [181]:
print(df["Stroma_Clear_Area_Diameter_mm"].describe())

count    555.000000
mean       8.417748
std        0.571562
min        6.000000
25%        8.000000
50%        8.500000
75%        9.000000
max       10.000000
Name: Stroma_Clear_Area_Diameter_mm, dtype: float64


In [183]:
for col in [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Stroma_Is_Clear ---
Stroma_Is_Clear
True     577
<NA>     467
False     11
Name: count, dtype: int64

--- Stroma_Is_Compact ---
Stroma_Is_Compact
<NA>     467
False    358
True     230
Name: count, dtype: int64

--- Stroma_Has_Arcus ---
Stroma_Has_Arcus
True     486
<NA>     467
False    102
Name: count, dtype: int64

--- Stroma_Has_Edema ---
Stroma_Has_Edema
False    578
<NA>     467
True      10
Name: count, dtype: int64

--- Stroma_Arcus_Severity ---
Stroma_Arcus_Severity
NaN            467
Unspecified    437
None           102
Mild            43
Moderate         6
Name: count, dtype: int64

--- Stroma_Scar_Type ---
Stroma_Scar_Type
None          555
<NA>          467
Other scar     27
Phaco scar      6
Name: count, dtype: int64


In [184]:
descemet_features = [
    "Descemet_Review_Flag",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
]

for col in descemet_features:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Descemet_Review_Flag ---
Descemet_Review_Flag
False    544
<NA>     469
True      42
Name: count, dtype: int64

--- Descemet_Has_Folds ---
Descemet_Has_Folds
True     535
<NA>     511
False      9
Name: count, dtype: int64

--- Descemet_Fold_Severity ---
Descemet_Fold_Severity
NaN            511
Mild           387
Unspecified    118
Moderate        21
Very mild       18
Name: count, dtype: int64

--- Descemet_Fold_Relative_Number ---
Descemet_Fold_Relative_Number
NaN                 511
Unspecified         261
Mild                252
Several              28
Moderate              2
Several/Moderate      1
Name: count, dtype: int64


In [185]:
endo_features = [
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
]

for col in endo_features:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Endo_Has_Stress_Lines ---
Endo_Has_Stress_Lines
True     499
<NA>     480
False     76
Name: count, dtype: int64

--- Endo_Stress_Line_Location ---
Endo_Stress_Line_Location
NaN                     480
Central                 243
Unspecified             174
Peripheral               67
None                     63
Central + Peripheral     28
Name: count, dtype: int64

--- Endo_Stress_Line_Amount ---
Endo_Stress_Line_Amount
NaN            480
Few            479
None            76
Unspecified     18
Several          2
Name: count, dtype: int64

--- Endo_Has_Cell_Dropout ---
Endo_Has_Cell_Dropout
False    482
<NA>     480
True      93
Name: count, dtype: int64

--- Endo_Cell_Dropout_Severity ---
Endo_Cell_Dropout_Severity
None        482
NaN         480
Mild         90
Moderate      3
Name: count, dtype: int64

--- Endo_Has_RBC ---
Endo_Has_RBC
False    545
<NA>     480
True      30
Name: count, dtype: int64

--- Endo_RBC_Severity ---
Endo_RBC_Severity
None           545
NaN           

In [186]:
clinical_features = [
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
    "Stroma_Clear_Area_Diameter_mm",
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
    "Descemet_Review_Flag",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
]

print("Number of clinical features:", len(clinical_features))
print("Missing feature columns:",
      [col for col in clinical_features if col not in df.columns])

print("\nSource columns still present:")
print([
    col for col in [
        "Epithelium_Findings",
        "Stroma_Findings",
        "Descemet_Findings",
        "Endothelium_Findings"
    ]
    if col in df.columns
])

print("\nDataset shape:", df.shape)

Number of clinical features: 25
Missing feature columns: []

Source columns still present:
['Epithelium_Findings', 'Stroma_Findings', 'Descemet_Findings', 'Endothelium_Findings']

Dataset shape: (1055, 113)


In [187]:
numeric_candidates = [
    "Form_ECD_cells_per_mm2",
    "Recipient_Age",
    "Donor_Graft_Size_mm",
    "Recipient_Bed_Size_mm",
    "Previous_Grafts",
]

for col in numeric_candidates:
    print(f"\n--- {col} ---")
    print(df[col].dtype)
    print(df[col].dropna().head(15).tolist())


--- Form_ECD_cells_per_mm2 ---
float64
[2132.0, 2212.0, 2538.0, 2584.0, 3115.0, 2950.0, 2993.0, 3049.0, 2833.0, 2780.0, 3049.0, 2710.0, 2625.0, 2833.0, 3012.0]

--- Recipient_Age ---
object
[48, 64, 53, 57, '2 y 3 m', '35', 80, '60 (as read)', 68, 47, 14, 69, '72 (as read)', 55, 21]

--- Donor_Graft_Size_mm ---
object
[8.25, 8, 8.5, 7.5, 8.5, 9, 7, 11, 7.5, 8, 8, '8-8.5', '8 (as read)', 8.25, 8]

--- Recipient_Bed_Size_mm ---
object
[8, 8.5, 8, 8, 8.25, 7.5, 7, 8.5, 8.25, '8 (as read)', '8 (as read)', 7.5, 9, '11.5 (as read)', 8.5]

--- Previous_Grafts ---
object
['1', '2', '1', 'Not ticked (diagnosis implies >=1)', '1', '1', 'Not ticked', '1', 'Not ticked', '>=1 (repeat PK)', 0, 1, 0, 1, 2]


In [193]:
def clean_graft_size(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()

    # Ignore written fractions such as "2/3 cornea"
    if re.search(r"\d+\s*/\s*\d+", text):
        return pd.NA

    # Handle ranges, including annotations such as "10-11 (as read)"
    range_match = re.search(
        r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)",
        text
    )

    if range_match:
        low = float(range_match.group(1))
        high = float(range_match.group(2))
        return (low + high) / 2

    # Handle single numeric values, including "(as read)"
    match = re.search(r"\b\d+(?:\.\d+)?\b", text)

    if match:
        return float(match.group())

    return pd.NA


df["Donor_Graft_Size_Clean_mm"] = (
    df["Donor_Graft_Size_mm"]
    .apply(clean_graft_size)
)

df["Donor_Graft_Size_Clean_mm"] = pd.to_numeric(
    df["Donor_Graft_Size_Clean_mm"],
    errors="coerce"
)

In [194]:
df.loc[
    df["Donor_Graft_Size_mm"].astype(str).str.contains(
        r"-|/|\(|as read|as written",
        case=False,
        na=False
    ),
    ["Tissue_ID", "Donor_Graft_Size_mm", "Donor_Graft_Size_Clean_mm"]
]

,Tissue_ID,Donor_Graft_Size_mm,Donor_Graft_Size_Clean_mm
27,NEB-2020-09-12680 R,8-8.5,8.25
28,NEB-2020-09-12681 L,8 (as read),8.00
37,NEB-2020-09-12690 L,11 (as read),11.00
57,NEB-2020-10-12710 L,10-11 (as read),10.50
60,NEB-2020-10-12713 R,8.5 (as read),8.50
114,NEB-2020-11-12816 R,8 (as read),8.00
233,NEB-2024-05-16063 R,14 (as read),14.00
241,NEB-2024-05-16071 R,14 (as read),14.00
284,NEB-2024-06-16115 L,11 (as read),11.00
288,NEB-2024-06-16119 R,8.5 (as read),8.50


In [196]:
print(df["Donor_Graft_Size_Clean_mm"].describe())

count    232.000000
mean       8.695259
std        1.282027
min        6.000000
25%        8.000000
50%        8.250000
75%        9.000000
max       14.000000
Name: Donor_Graft_Size_Clean_mm, dtype: float64


In [198]:
print(df.loc[
    df["Donor_Graft_Size_mm"].astype(str).str.contains(
        r"2/3|10-11",
        case=False,
        na=False
    ),
    ["Tissue_ID", "Donor_Graft_Size_mm", "Donor_Graft_Size_Clean_mm"]
])


               Tissue_ID      Donor_Graft_Size_mm  Donor_Graft_Size_Clean_mm
57   NEB-2020-10-12710 L          10-11 (as read)                       10.5
570  NEB-2025-11-17628 R  2/3 cornea (as written)                        NaN


In [199]:
df.loc[
    df["Recipient_Bed_Size_mm"].astype(str).str.contains(
        r"-|\(|as read|as written",
        case=False,
        na=False
    ),
    ["Tissue_ID", "Recipient_Bed_Size_mm"]
]

,Tissue_ID,Recipient_Bed_Size_mm
28,NEB-2020-09-12681 L,8 (as read)
31,NEB-2020-09-12684 R,8 (as read)
37,NEB-2020-09-12690 L,11.5 (as read)
59,NEB-2020-10-12712 L,8.5 (as read)
80,NEB-2020-10-12735 R,10 (as read)
124,NEB-2020-11-12829 R,11 (as read)
233,NEB-2024-05-16063 R,12 (as read)
241,NEB-2024-05-16071 R,12 (as read)
287,NEB-2024-06-16118 L,8.5 (as read)
321,NEB-2024-08-16254 L,11 (as read)


In [200]:
print(df["Recipient_Bed_Size_mm"].value_counts(dropna=False).head(30))

Recipient_Bed_Size_mm
NaN                           862
8                              67
8.5                            21
8.25                           19
7.5                            12
7                              12
9                              10
7.75                            8
9.5                             6
10                              6
12                              5
11                              5
10.5                            3
8 (as read)                     2
8.5 (as read)                   2
11 (as read)                    2
8.75                            2
12 (as read)                    2
11.5 (as read)                  1
6.5                             1
10 (as read)                    1
Freehand                        1
14                              1
9 (as read)                     1
7.25                            1
8.7 (as read)                   1
2/3 of cornea (as written)      1
Name: count, dtype: int64


In [201]:
def clean_bed_size(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()

    # Written fraction, not a numeric diameter
    if re.search(r"\d+\s*/\s*\d+", text):
        return pd.NA

    # Numeric value, including "(as read)"
    match = re.search(r"\b\d+(?:\.\d+)?\b", text)

    if match:
        return float(match.group())

    return pd.NA


df["Recipient_Bed_Size_Clean_mm"] = (
    df["Recipient_Bed_Size_mm"]
    .apply(clean_bed_size)
)

df["Recipient_Bed_Size_Clean_mm"] = pd.to_numeric(
    df["Recipient_Bed_Size_Clean_mm"],
    errors="coerce"
)

df["Recipient_Bed_Size_Clean_mm"].describe()

count    191.000000
mean       8.503665
std        1.198491
min        6.500000
25%        8.000000
50%        8.000000
75%        8.500000
max       14.000000
Name: Recipient_Bed_Size_Clean_mm, dtype: float64

In [203]:
print(df.loc[
    df["Recipient_Bed_Size_mm"].astype(str).str.contains(
        r"freehand|2/3",
        case=False,
        na=False
    ),
    ["Tissue_ID", "Recipient_Bed_Size_mm", "Recipient_Bed_Size_Clean_mm"]
])

               Tissue_ID       Recipient_Bed_Size_mm  \
238  NEB-2024-05-16068 L                    Freehand   
570  NEB-2025-11-17628 R  2/3 of cornea (as written)   

     Recipient_Bed_Size_Clean_mm  
238                          NaN  
570                          NaN  


In [204]:
df["Recipient_Bed_Size_Clean_mm"].describe()

count    191.000000
mean       8.503665
std        1.198491
min        6.500000
25%        8.000000
50%        8.000000
75%        8.500000
max       14.000000
Name: Recipient_Bed_Size_Clean_mm, dtype: float64

In [206]:
print(df.loc[
    df["Recipient_Bed_Size_mm"].astype(str).str.contains(
        r"\(as read\)",
        case=False,
        na=False
    ),
    ["Tissue_ID", "Recipient_Bed_Size_mm", "Recipient_Bed_Size_Clean_mm"]
])

               Tissue_ID Recipient_Bed_Size_mm  Recipient_Bed_Size_Clean_mm
28   NEB-2020-09-12681 L           8 (as read)                          8.0
31   NEB-2020-09-12684 R           8 (as read)                          8.0
37   NEB-2020-09-12690 L        11.5 (as read)                         11.5
59   NEB-2020-10-12712 L         8.5 (as read)                          8.5
80   NEB-2020-10-12735 R          10 (as read)                         10.0
124  NEB-2020-11-12829 R          11 (as read)                         11.0
233  NEB-2024-05-16063 R          12 (as read)                         12.0
241  NEB-2024-05-16071 R          12 (as read)                         12.0
287  NEB-2024-06-16118 L         8.5 (as read)                          8.5
321  NEB-2024-08-16254 L          11 (as read)                         11.0
463  NEB-2025-09-17515 R           9 (as read)                          9.0
478  NEB-2025-09-17530 R         8.7 (as read)                          8.7


In [207]:
print(df["Recipient_Bed_Size_Clean_mm"].describe())

count    191.000000
mean       8.503665
std        1.198491
min        6.500000
25%        8.000000
50%        8.000000
75%        8.500000
max       14.000000
Name: Recipient_Bed_Size_Clean_mm, dtype: float64


In [208]:
print(df["Previous_Grafts"].value_counts(dropna=False))

Previous_Grafts
NaN                                                982
1                                                   30
1                                                   10
Not ticked                                           8
Not recorded                                         5
2                                                    4
Not ticked (diagnosis implies >=1)                   2
0                                                    2
2                                                    2
None (as read)                                       2
>=1 (repeat PK)                                      1
None (ticked - but diagnosis is rejected graft)      1
None marked and "1" written (as read)                1
3                                                    1
None marked and "2" written (as read)                1
1 (per letter; not ticked on request form)           1
2 (as read)                                          1
None (debulking + conjunctival flap graft done)  

In [210]:
previous_graft_special = [
    "Not ticked",
    "Not recorded",
    "Not ticked (diagnosis implies >=1)",
    "None (as read)",
    ">=1 (repeat PK)",
    "None (ticked - but diagnosis is rejected graft)",
    'None marked and "1" written (as read)',
    'None marked and "2" written (as read)',
    "1 (per letter; not ticked on request form)",
    "2 (as read)",
    "None (debulking + conjunctival flap graft done)",
]

print(df.loc[
    df["Previous_Grafts"].isin(previous_graft_special),
    ["Tissue_ID", "Previous_Grafts", "Tissue_Grade_Clean", "Disease_Label"]
])

               Tissue_ID                                  Previous_Grafts  \
51   NEB-2020-09-12704 L               Not ticked (diagnosis implies >=1)   
59   NEB-2020-10-12712 L                                       Not ticked   
82   NEB-2020-10-12737 L                                       Not ticked   
179  NEB-2022-10-14306 R                                  >=1 (repeat PK)   
244  NEB-2024-05-16074 R               Not ticked (diagnosis implies >=1)   
293  NEB-2024-06-16124 R  None (ticked - but diagnosis is rejected graft)   
297  NEB-2024-06-16128 L            None marked and "1" written (as read)   
303  NEB-2024-06-16134 R            None marked and "2" written (as read)   
378  NEB-2024-12-16628 L                                       Not ticked   
390  NEB-2024-12-16640 R                                       Not ticked   
393  NEB-2024-12-16644 L                                     Not recorded   
420  NEB-2025-09-17472 L       1 (per letter; not ticked on request form)   

In [211]:
previous_grafts_mapping = {
    "Not ticked": "Unknown",
    "Not recorded": "Unknown",
    "Not ticked (diagnosis implies >=1)": ">=1",
    ">=1 (repeat PK)": ">=1",
    "None (as read)": "0",
    "None (ticked - but diagnosis is rejected graft)": "0",
    'None marked and "1" written (as read)': "1",
    'None marked and "2" written (as read)': "2",
    "1 (per letter; not ticked on request form)": "1",
    "2 (as read)": "2",
    "None (debulking + conjunctival flap graft done)": "0",
}

In [213]:
print(df["Previous_Grafts"].map(type).value_counts())

Previous_Grafts
<class 'float'>    982
<class 'str'>       59
<class 'int'>       14
Name: count, dtype: int64


In [215]:
def clean_previous_grafts(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().lower()

    # Explicit / documented values
    if text in {"0", "none (as read)",
                "none (ticked - but diagnosis is rejected graft)",
                "none (debulking + conjunctival flap graft done)"}:
        return "0"

    if text in {"1", 'none marked and "1" written (as read)',
                "1 (per letter; not ticked on request form)"}:
        return "1"

    if text in {"2", 'none marked and "2" written (as read)',
                "2 (as read)"}:
        return "2"

    if text == "3":
        return "3"

    if text in {
        "not ticked",
        "not recorded",
    }:
        return "Unknown"

    if text in {
        "not ticked (diagnosis implies >=1)",
        ">=1 (repeat pk)",
    }:
        return ">=1"

    return "Unknown"


df["Previous_Grafts_Clean"] = (
    df["Previous_Grafts"]
    .apply(clean_previous_grafts)
)

print(df["Previous_Grafts_Clean"].value_counts(dropna=False))

Previous_Grafts_Clean
NaN        982
1           42
Unknown     13
2            8
0            6
>=1          3
3            1
Name: count, dtype: int64


In [216]:
print(
    "Original non-missing:",
    df["Previous_Grafts"].notna().sum()
)

print(
    "Clean non-missing:",
    df["Previous_Grafts_Clean"].notna().sum()
)

print(
    "Unexpected clean values:",
    sorted(
        set(df["Previous_Grafts_Clean"].dropna().unique())
        - {"0", "1", "2", "3", ">=1", "Unknown"}
    )
)

Original non-missing: 73
Clean non-missing: 73
Unexpected clean values: []


In [217]:
candidate_columns = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Form_ECD_cells_per_mm2",
    "Donor_Age",
    "Recipient_Age_Clean",
    "Donor_Graft_Size_Clean_mm",
    "Recipient_Bed_Size_Clean_mm",
    "Previous_Grafts_Clean",
    "Clinical_Findings_Available",
]

for col in candidate_columns:
    print(
        f"{col:35} | "
        f"dtype={df[col].dtype} | "
        f"non-null={df[col].notna().sum()}"
    )

ECD_CD_cells_per_mm2                | dtype=float64 | non-null=952
CV                                  | dtype=float64 | non-null=952
HEX_percent                         | dtype=float64 | non-null=952
SD                                  | dtype=float64 | non-null=951
Total_Cell_Count                    | dtype=float64 | non-null=952
Total_Area_um2                      | dtype=float64 | non-null=952
Form_ECD_cells_per_mm2              | dtype=float64 | non-null=575
Donor_Age                           | dtype=float64 | non-null=1050
Recipient_Age_Clean                 | dtype=float64 | non-null=295
Donor_Graft_Size_Clean_mm           | dtype=float64 | non-null=232
Recipient_Bed_Size_Clean_mm         | dtype=float64 | non-null=191
Previous_Grafts_Clean               | dtype=str | non-null=73
Clinical_Findings_Available         | dtype=int64 | non-null=1055


In [220]:
print(df["Donor_Age"].describe())

count    1050.000000
mean       59.672381
std        10.901732
min        15.000000
25%        54.000000
50%        62.000000
75%        68.000000
max        78.000000
Name: Donor_Age, dtype: float64


In [221]:
print(df.loc[
    (df["Donor_Age"] < 0) | (df["Donor_Age"] > 120),
    ["Tissue_ID", "Donor_Age"]
])

Empty DataFrame
Columns: [Tissue_ID, Donor_Age]
Index: []


In [222]:
grading_numeric_candidates = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Form_ECD_cells_per_mm2",
    "Donor_Age",
]

missing_summary = pd.DataFrame({
    "Missing_Count": df[grading_numeric_candidates].isna().sum(),
    "Missing_Percent": (
        df[grading_numeric_candidates].isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing_Percent",
    ascending=False
)

,Missing_Count,Missing_Percent
Form_ECD_cells_per_mm2,480,45.50
SD,104,9.86
CV,103,9.76
ECD_CD_cells_per_mm2,103,9.76
HEX_percent,103,9.76
Total_Cell_Count,103,9.76
Total_Area_um2,103,9.76
Donor_Age,5,0.47


In [225]:
konan_cols = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

print(df[konan_cols].isna().value_counts())

ECD_CD_cells_per_mm2  CV     HEX_percent  SD     Total_Cell_Count  Total_Area_um2
False                 False  False        False  False             False             951
True                  True   True         True   True              True              103
False                 False  False        True   False             False               1
Name: count, dtype: int64


In [226]:
print(df.loc[
    df[konan_cols].isna().all(axis=1),
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "Konan_Report_File",
        "SP_Image_File",
        "Cell_Image_File"
    ]
].head(20))

                       Tissue_ID Tissue_Grade_Clean Konan_Report_File  \
0   12346 R (year/month unknown)                NaN               NaN   
30           NEB-2020-09-12683 L                 A+               NaN   
49           NEB-2020-09-12702 R                 A+               NaN   
50           NEB-2020-09-12703 L                 A+               NaN   
51           NEB-2020-09-12704 L                  A               NaN   
52           NEB-2020-09-12705 R                  A               NaN   
53           NEB-2020-10-12706 R                  A               NaN   
54           NEB-2020-10-12707 L                  A               NaN   
55           NEB-2020-10-12708 L                  B               NaN   
56           NEB-2020-10-12709 R                  B               NaN   
57           NEB-2020-10-12710 L                 A+               NaN   
58           NEB-2020-10-12711 R                 A+               NaN   
59           NEB-2020-10-12712 L                  B

In [227]:
konan_missing = df[konan_cols].isna().all(axis=1)

df.loc[
    konan_missing,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "Tissue_Grade_Source",
        "Dataset_Part",
        "Konan_Report_File",
        "SP_Image_File",
        "Cell_Image_File"
    ]
].isna().sum()

Tissue_ID               0
Tissue_Grade_Clean      7
Tissue_Grade_Source     6
Dataset_Part            0
Konan_Report_File      93
SP_Image_File          88
Cell_Image_File        89
dtype: int64

In [229]:
print(df.loc[
    konan_missing,
    "Tissue_Grade_Clean"
].value_counts(dropna=False))

Tissue_Grade_Clean
A+     40
A      38
B      18
NaN     7
Name: count, dtype: int64


In [230]:
konan_missing = df[konan_cols].isna().all(axis=1)

df.loc[konan_missing, "Tissue_ID"].astype(str).str.extract(
    r"NEB-(\d{4})-"
)[0].value_counts().sort_index()

0
2020    56
2021     1
2022     5
2023     5
2024    19
2025    15
Name: count, dtype: int64

In [232]:
konan_available = ~df[konan_cols].isna().all(axis=1)

print(pd.DataFrame({
    "Konan_Available": df.loc[konan_available, "Tissue_ID"]
        .astype(str).str.extract(r"NEB-(\d{4})-")[0].value_counts(),
    "Konan_Missing": df.loc[konan_missing, "Tissue_ID"]
        .astype(str).str.extract(r"NEB-(\d{4})-")[0].value_counts()
}).fillna(0).astype(int))

      Konan_Available  Konan_Missing
0                                   
2020               88             56
2021               12              1
2022               58              5
2023              459              5
2024              159             19
2025              176             15


In [234]:
form_ecd_missing = df["Form_ECD_cells_per_mm2"].isna()

print(df.loc[
    form_ecd_missing,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "Tissue_Grade_Source",
        "Dataset_Part",
        "ECD_CD_cells_per_mm2",
        "Form_ECD_cells_per_mm2"
    ]
].head(20))

                        Tissue_ID Tissue_Grade_Clean Tissue_Grade_Source  \
0    12346 R (year/month unknown)                NaN                 NaN   
61            NEB-2020-10-12715 R                 A+       Eye-bank form   
64            NEB-2020-10-12718 L                 A+       Eye-bank form   
87            NEB-2020-10-12742 R                  A       Eye-bank form   
97            NEB-2020-10-12761 R                NaN                 NaN   
174  14302 L (year/month unknown)                NaN                 NaN   
285           NEB-2024-06-16116 L                NaN                 NaN   
286           NEB-2024-06-16117 R                NaN                 NaN   
306           NEB-2024-06-16137 L                NaN                 NaN   
319           NEB-2024-08-16252 L                 A+       Eye-bank form   
325           NEB-2024-08-16258 L                 A+       Eye-bank form   
367           NEB-2024-08-16300 R                  B       Eye-bank form   
368         

In [236]:
print(df.loc[
    form_ecd_missing,
    "Tissue_Grade_Clean"
].value_counts(dropna=False))

Tissue_Grade_Clean
A+     260
A      120
B       92
NaN      8
Name: count, dtype: int64


In [237]:
ecd_source_overlap = pd.DataFrame({
    "Konan_ECD_Available": df["ECD_CD_cells_per_mm2"].notna(),
    "Form_ECD_Available": df["Form_ECD_cells_per_mm2"].notna()
})

print(ecd_source_overlap.value_counts())

Konan_ECD_Available  Form_ECD_Available
True                 True                  490
                     False                 462
False                True                   85
                     False                  18
Name: count, dtype: int64


In [238]:
df.loc[
    df["SD"].isna() &
    df["ECD_CD_cells_per_mm2"].notna(),
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2"
    ]
]

,Tissue_ID,Tissue_Grade_Clean,ECD_CD_cells_per_mm2,CV,HEX_percent,SD,Total_Cell_Count,Total_Area_um2
273,NEB-2024-05-16103 R,B,2203.0,50.0,42.0,NaN,99.0,44938.0


In [239]:
df.loc[
    df["ECD_CD_cells_per_mm2"].notna() &
    df["SD"].isna(),
    "SD"
] = 229

In [240]:
df["SD"].isna().sum()

np.int64(103)

In [241]:
grading_numeric_candidates = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

In [243]:
grading_numeric_candidates = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

print(df[grading_numeric_candidates].describe().T)

                       count          mean          std     min      25%  \
ECD_CD_cells_per_mm2   952.0   2592.400210   438.998119   676.0   2336.0   
CV                     952.0     38.130252     4.993776    24.0     35.0   
HEX_percent            952.0     54.032563     6.437436    27.0     50.0   
SD                     952.0    153.080882    47.739818    86.0    127.0   
Total_Cell_Count       952.0    107.018908    16.351349    10.0    104.0   
Total_Area_um2         952.0  42265.138655  9122.697304  5071.0  37512.5   
Donor_Age             1050.0     59.672381    10.901732    15.0     54.0   

                          50%       75%       max  
ECD_CD_cells_per_mm2   2591.0   2857.00    4484.0  
CV                       38.0     41.00      74.0  
HEX_percent              54.0     58.00      76.0  
SD                      144.0    164.25     753.0  
Total_Cell_Count        108.0    113.00     204.0  
Total_Area_um2        41464.5  46336.50  100692.0  
Donor_Age                62

In [244]:
outlier_check_cols = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

for col in outlier_check_cols:
    print(f"\n--- {col} ---")
    print(
        df.loc[
            df[col].notna(),
            ["Tissue_ID", "Tissue_Grade_Clean", col]
        ]
        .sort_values(col)
        .head(5)
    )
    print(
        df.loc[
            df[col].notna(),
            ["Tissue_ID", "Tissue_Grade_Clean", col]
        ]
        .sort_values(col, ascending=False)
        .head(5)
    )


--- ECD_CD_cells_per_mm2 ---
               Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2
105  NEB-2020-11-12807 L                  B                 676.0
562  NEB-2025-11-17620 R                  B                 788.0
161  NEB-2022-04-13824 R                  B                 905.0
823  NEB-2023-06-15129 R                  B                 951.0
881  NEB-2023-07-15250 R                  B                1008.0
               Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2
298  NEB-2024-06-16129 R                  A                4484.0
716  NEB-2023-03-14912 R                 A+                4032.0
129  NEB-2020-11-12835 R                  A                3831.0
205  NEB-2022-10-14333 R                  A                3774.0
971  NEB-2023-10-15440 R                  A                3759.0

--- CV ---
               Tissue_ID Tissue_Grade_Clean    CV
967  NEB-2023-10-15432 R                 A+  24.0
644  NEB-2023-02-14764 R                  B  24.0
529  NEB-2025-10

In [245]:
print(df.loc[
    df["Total_Cell_Count"] <= 20,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2"
    ]
].sort_values("Total_Cell_Count"))

               Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2    CV  \
644  NEB-2023-02-14764 R                  B                1972.0  24.0   
120  NEB-2020-11-12824 L                  B                2212.0  31.0   
797  NEB-2023-05-15073 R                  B                2008.0  29.0   
105  NEB-2020-11-12807 L                  B                 676.0  51.0   
795  NEB-2023-05-15069 R                  B                2165.0  25.0   
683  NEB-2023-03-14837 R                  B                1370.0  30.0   

     HEX_percent     SD  Total_Cell_Count  Total_Area_um2  
644         60.0  121.0              10.0          5071.0  
120         33.0  138.0              12.0          5418.0  
797         56.0  145.0              16.0          7961.0  
105         27.0  753.0              18.0         26625.0  
795         38.0  117.0              18.0          8313.0  
683         68.0  218.0              19.0         13871.0  


In [246]:
df.shape

(1055, 116)

In [247]:
df.columns.tolist()

['Tissue_ID',
 'Eye',
 'Image_ID',
 'Disease_Label',
 'Disease_Severity',
 'ECD_CD_cells_per_mm2',
 'CV',
 'HEX_percent',
 'SD',
 'Total_Cell_Count',
 'Total_Area_um2',
 'Epithelium_Findings',
 'Epithelium_Severity',
 'Stroma_Findings',
 'Stroma_Severity',
 'Descemet_Findings',
 'Descemet_Severity',
 'Endothelium_Findings',
 'Endothelium_Severity',
 'Additional_Findings',
 'Tissue_Grade',
 'Overall_Donor_Quality',
 'Transplant_Suitability',
 'Suitability_Reason',
 'Primary_COD',
 'Donor_Age',
 'Tissue_Serial',
 'Linkage_Status',
 'Source_PDF',
 'PDF_Pages',
 'Konan_Report_File',
 'SP_Image_File',
 'Cell_Image_File',
 'Konan_Raw_CorneaID',
 'Konan_Exam_Date',
 'Konan_Mean_Cell_Area_um2',
 'Konan_Comments',
 'Donor_Sex',
 'Admitting_Diagnosis',
 'Death_Date_Time',
 'Preservation_Date_Time',
 'Death_to_Preservation_Time',
 'Storage_Medium',
 'Serology_HBsAg',
 'Serology_HCV',
 'Serology_HIV',
 'Serology_VDRL',
 'Lens_Status',
 'Form_ECD_cells_per_mm2',
 'Intended_Use',
 'Tissue_Outcome',


In [259]:
feature_role_audit = {
    "Target": [
        "Tissue_Grade_Clean",
    ],

    "Primary_Clinical_Numeric": [
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2",
        "Donor_Age",
    ],

    "Structured_Clinical_Findings": [
        "Clinical_Findings_Available",
        "Stroma_Clear_Area_Diameter_mm",
        "Stroma_Is_Clear",
        "Stroma_Is_Compact",
        "Stroma_Has_Arcus",
        "Stroma_Has_Edema",
        "Stroma_Has_Scar",
        "Stroma_Arcus_Severity",
        "Stroma_Scar_Type",
        "Descemet_Review_Flag",
        "Descemet_Has_Folds",
        "Descemet_Fold_Severity",
        "Descemet_Fold_Relative_Number",
        "Endo_Has_Stress_Lines",
        "Endo_Stress_Line_Location",
        "Endo_Stress_Line_Amount",
        "Endo_Has_Cell_Dropout",
        "Endo_Cell_Dropout_Severity",
        "Endo_Has_RBC",
        "Endo_RBC_Severity",
        "Epi_Has_Exposure",
        "Epi_Exposure_Severity",
        "Epi_Exposure_Location",
        "Epi_Has_Defect",
        "Epi_Has_Haze",
        "Epi_Haze_Severity",
        "Epi_Has_Arcus",
    ],

    "Secondary_or_Contextual": [
        "Form_ECD_cells_per_mm2",
        "Recipient_Age_Clean",
        "Donor_Graft_Size_Clean_mm",
        "Recipient_Bed_Size_Clean_mm",
        "Previous_Grafts_Clean",
],

    "Identifiers_and_Metadata": [
        "Tissue_ID",
        "Eye",
        "Image_ID",
        "Tissue_Serial",
        "Dataset_Part",
        "Tissue_Grade_Source",
        "Disease_Label_Source",
        "Source_PDF",
        "PDF_Pages",
        "Konan_Report_File",
        "SP_Image_File",
        "Cell_Image_File",
    ],

    "Leakage_or_Downstream": [
        "Overall_Donor_Quality",
        "Transplant_Suitability",
        "Suitability_Reason",
        "Tissue_Outcome",
        "Recipient_Age",
        "Recipient_Eye",
        "Recipient_Indication",
        "Request_Indication",
        "Primary_Procedure",
        "Secondary_Procedures",
        "Surgery_Date",
        "Requested_Grade",
        "Planned_Procedure",
        "Grade_Request_Met",
        "Review_Notes",
        "Additional_Findings",
    ],

    "Raw_Source_Columns": [
        "Tissue_Grade",
        "Disease_Label",
        "Epithelium_Findings",
        "Stroma_Findings",
        "Descemet_Findings",
        "Endothelium_Findings",
        "Recipient_Age",
        "Donor_Graft_Size_mm",
        "Recipient_Bed_Size_mm",
        "Previous_Grafts",
],
}

In [260]:
feature_role_audit["Donor_and_Context"] = [
    "Donor_Sex",
    "Admitting_Diagnosis",
    "Primary_COD",
    "Lens_Status",
    "Storage_Medium",
    "Serology_HBsAg",
    "Serology_HCV",
    "Serology_HIV",
    "Serology_VDRL",
]

feature_role_audit["Clinical_Source_Metadata"] = [
    "Disease_Severity",
    "Epithelium_Severity",
    "Stroma_Severity",
    "Descemet_Severity",
    "Endothelium_Severity",
    "Needs_Review",
    "Linkage_Status",
]

feature_role_audit["Konan_Source_Metadata"] = [
    "Konan_Raw_CorneaID",
    "Konan_Exam_Date",
    "Konan_Mean_Cell_Area_um2",
    "Konan_Comments",
]

feature_role_audit["Request_and_Surgery_Metadata"] = [
    "Intended_Use",
    "Suture_Technique",
    "Intraop_Complications",
    "Death_to_Surgery_Days",
    "Request_Reg_No",
    "Request_Issue_No",
    "Request_Date",
    "Proposed_Op_Date",
    "Request_Urgency",
    "Request_Indication_Type",
    "Request_to_Surgery_Days",
    "Requesting_Surgeon",
    "Requesting_Hospital",
    "Recipient_Sex",
    "VA_Operative_Eye",
    "VA_Fellow_Eye",
]

feature_role_audit["Administrative_and_Temporal_Metadata"] = [
    "Death_Date_Time",
    "Death_to_Preservation_Time",
    "Preservation_Date_Time",
]

In [261]:
audit_columns = [
    col
    for columns in feature_role_audit.values()
    for col in columns
]

missing_from_audit = sorted(
    set(df.columns) - set(audit_columns)
)

missing_from_audit

[]

In [264]:
[col for category, columns in feature_role_audit.items()
 for col in columns
 if col == "Recipient_Age"]

['Recipient_Age', 'Recipient_Age']

In [266]:
[
    category
    for category, columns in feature_role_audit.items()
    if "Recipient_Age" in columns
]

['Leakage_or_Downstream', 'Raw_Source_Columns']

In [267]:
from collections import Counter

column_counts = Counter(audit_columns)

duplicates_in_audit = {
    col: count
    for col, count in column_counts.items()
    if count > 1
}

duplicates_in_audit

{'Recipient_Age': 2}

In [268]:
feature_role_audit["Raw_Source_Columns"].remove("Recipient_Age")

In [269]:
duplicates_in_audit = {
    col: count
    for col, count in Counter(
        col
        for columns in feature_role_audit.values()
        for col in columns
    ).items()
    if count > 1
}

duplicates_in_audit

{}

In [270]:
missing_from_audit = sorted(
    set(df.columns) -
    {
        col
        for columns in feature_role_audit.values()
        for col in columns
    }
)

missing_from_audit

[]